# TAVE 18기 DRAG 스터디 - 5주차 텍스트→이미지 검색 (SigLIP + FAISS)

smol-vision의 [Image Search with MetaCLIP2](https://github.com/merveenoyan/smol-vision/blob/main/Image_Search_with_MetaCLIP2.ipynb) 노트북을 베이스로, 모델을 **SigLIP**으로 바꿔서 텍스트로 이미지를 찾는 검색 시스템을 만든다.

**흐름 (원본 노트북과 동일)**
1. **오프라인 인덱싱**: 모든 이미지를 모델의 이미지 쪽에 통과시켜 임베딩을 만들고 벡터DB(FAISS)에 저장. 한 번만 하면 되고 시간이 걸림
2. **온라인 추론**: 텍스트 질의를 모델의 텍스트 쪽에 통과시켜 질의 임베딩을 만듦
3. **유사도 계산 & 조회**: 질의 임베딩과 이미지 임베딩의 유사도로 가장 가까운 이미지를 찾고, 인덱스 번호로 원래 이미지를 조회

**원본(MetaCLIP2)에서 바꾼 점**

| 항목 | 원본 | 이 노트북 | 이유 |
|---|---|---|---|
| 모델 | `facebook/metaclip-2-worldwide-giant` | `google/siglip-base-patch16-256-multilingual` | 4주차에서 배운 SigLIP, 한국어 질의용 다국어 모델 |
| 데이터 | `merve/food` (16장) | Flickr8k 등에서 수백 장 (실패하면 `merve/food`) | 5주차 과제: 이미지 데이터셋 대상 검색 |
| 텍스트 입력 | `processor.tokenizer([prompt])` | `processor(text=..., padding="max_length")` | SigLIP은 학습 때처럼 64토큰 패딩 필수 |
| 임베딩 꺼내기 | 결과가 바로 벡터 | `.pooler_output` 처리 | transformers 5.x는 출력 객체를 돌려줌 |
| 인덱스 | `IndexFlatL2(1280)` | `IndexFlatIP(768)` | 정규화한 벡터의 내적 = 코사인 유사도 (클수록 비슷) |
| 점수 | L2 거리 | 코사인 + SigLIP '짝일 확률' | 4주차: `sigmoid(t·cos + b)`로 관련 없는 결과 거르기 |
| 추가 | - | 인덱스 저장/불러오기, Recall@K, Gradio 데모 | 매번 다시 계산하지 않기, 검색 정확도 숫자로 보기 |

**실행 방법**: `런타임 → 런타임 유형 변경 → GPU(T4)` 선택 후 위에서부터 차례대로 실행 (CPU에서도 돌아가지만 인덱싱이 느림)

In [ ]:
!pip install -q -U faiss-cpu datasets transformers sentencepiece

## 0. 설정

In [ ]:
import os, json, time
import numpy as np
import torch
from PIL import Image

MODEL_ID = "google/siglip-base-patch16-256-multilingual"  # 영어만 쓸 거면 "google/siglip-base-patch16-224"

# 이미지 데이터셋: 위에서부터 차례로 시도하고, 안 되면 다음 것으로 넘어간다
DATASETS = [
    ("jxie/flickr8k", "test"),        # 일상 사진 8천 장 + 캡션 5개씩
    ("nlphuji/flickr30k", "test"),    # Flickr8k 확장판
    ("merve/food", "train"),          # 원본 노트북 데이터 (16장, 캡션 없음)
]
N_IMAGES = 500       # 인덱싱할 이미지 수 (처음엔 300~500장 추천)
BATCH = 32           # 한 번에 임베딩할 이미지 수
TOP_K = 5            # 검색 결과로 보여줄 이미지 수
MIN_PROB = 0.01      # SigLIP '짝일 확률'이 이보다 낮으면 '관련 없음'으로 표시

WORKDIR = "/content/drag_week5" if os.path.isdir("/content") else "./drag_week5"
IMG_DIR = f"{WORKDIR}/images"
INDEX_PATH = f"{WORKDIR}/siglip.index"
META_PATH = f"{WORKDIR}/meta.json"
os.makedirs(IMG_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 1. 이미지 데이터셋 불러오기
전체를 내려받지 않고 `streaming=True`로 앞에서부터 `N_IMAGES`장만 가져온다. 이미지는 파일로 저장해 두고(나중에 다시 보여주기용), 캡션이 있으면 같이 저장한다(Recall@K용).

In [ ]:
from datasets import load_dataset


def find_columns(row):
    """이미지 컬럼과 캡션 컬럼 이름을 자동으로 찾는다 (데이터셋마다 이름이 달라서)."""
    img_col = next(k for k, v in row.items() if isinstance(v, Image.Image))
    cap_cols = [k for k in row if any(w in k.lower() for w in ("caption", "sentence", "text"))]
    return img_col, cap_cols


def load_images(n):
    for ds_id, split in DATASETS:
        try:
            ds = load_dataset(ds_id, split=split, streaming=True)
            rows = list(ds.take(n))
            img_col, cap_cols = find_columns(rows[0])
            print(f"✅ {ds_id} ({split}) · 이미지 컬럼 '{img_col}' · 캡션 컬럼 {cap_cols}")
            return ds_id, rows, img_col, cap_cols
        except Exception as e:
            print(f"❌ {ds_id}: {type(e).__name__}: {str(e)[:120]}")
    raise RuntimeError("불러올 수 있는 데이터셋이 없습니다. DATASETS 목록을 확인하세요.")


DATASET_ID, rows, img_col, cap_cols = load_images(N_IMAGES)

meta = []  # 인덱스 번호 i ↔ meta[i] (파일 경로, 캡션)
for i, row in enumerate(rows):
    path = f"{IMG_DIR}/{i:05d}.jpg"
    row[img_col].convert("RGB").save(path, quality=90)
    caps = []
    for c in cap_cols:
        v = row[c]
        caps += v if isinstance(v, list) else [v]
    meta.append({"path": path, "captions": [str(c) for c in caps if c]})

print(f"이미지 {len(meta)}장 저장 · 캡션 있는 이미지 {sum(bool(m['captions']) for m in meta)}장")
meta[0]

## 2. 모델 불러오기
FAISS는 float32 벡터만 받기 때문에 원본 노트북처럼 모델도 float32로 불러온다.
`t`(온도)와 `b`(바이어스)는 4주차에서 배운 SigLIP의 학습된 값으로, 코사인 유사도를 '짝일 확률'로 바꿀 때 쓴다.

In [ ]:
from transformers import AutoProcessor, AutoModel

model = AutoModel.from_pretrained(MODEL_ID, dtype=torch.float32).to(device).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
T = model.logit_scale.exp().item()  # 온도 t
B = model.logit_bias.item()         # 바이어스 b
print(f"t = {T:.2f}, b = {B:.2f}")

## 3. 오프라인 인덱싱: 이미지 → 임베딩 → FAISS
- 원본의 `embed_metaclip()` 자리에 `embed_images()`를 쓴다. 여러 장을 묶어서(BATCH) 한 번에 처리해서 빠르다
- `faiss.normalize_L2()`로 길이를 1로 맞춘 뒤 `IndexFlatIP`(내적)에 넣는다 → 내적 = 코사인 유사도

In [ ]:
import faiss


def to_vector(out):
    """transformers 5.x는 출력 객체(.pooler_output), 4.x는 텐서를 돌려준다. 둘 다 numpy float32로."""
    out = getattr(out, "pooler_output", out)
    return np.float32(out.detach().cpu().numpy())


@torch.no_grad()
def embed_images(images):
    inputs = processor(images=images, return_tensors="pt").to(device)
    return to_vector(model.get_image_features(**inputs))


@torch.no_grad()
def embed_texts(texts):
    # important: SigLIP은 학습 때처럼 padding="max_length"(64토큰)로 넣어야 한다
    inputs = processor(text=texts, padding="max_length", truncation=True, return_tensors="pt").to(device)
    return to_vector(model.get_text_features(**inputs))


t0 = time.time()
vectors = []
for i in range(0, len(meta), BATCH):
    batch = [Image.open(m["path"]).convert("RGB") for m in meta[i:i + BATCH]]
    vectors.append(embed_images(batch))
    print(f"\r{min(i + BATCH, len(meta))}/{len(meta)}장 임베딩", end="")
vectors = np.vstack(vectors)
faiss.normalize_L2(vectors)

DIM = vectors.shape[1]  # base 모델은 768
index = faiss.IndexFlatIP(DIM)
index.add(vectors)
print(f"\n인덱스 완성: {index.ntotal}개 · {DIM}차원 · {time.time() - t0:.0f}초")

# 저장해 두면 다음엔 이 셀을 건너뛰고 '5. 인덱스 다시 불러오기'만 하면 된다
faiss.write_index(index, INDEX_PATH)
json.dump({"model": MODEL_ID, "dataset": DATASET_ID, "items": meta}, open(META_PATH, "w"), ensure_ascii=False)
print("저장:", INDEX_PATH, META_PATH)

## 4. 온라인 추론 + 조회: 텍스트 → 이미지 검색
- 질의를 텍스트 쪽으로 임베딩 → 정규화 → `index.search()`로 상위 k개
- 점수 두 가지를 같이 본다
  - **코사인 유사도**: 순위를 정하는 값 (클수록 비슷)
  - **짝일 확률** = `sigmoid(t · 코사인 + b)`: '관련 있음/없음'을 거르는 기준 (4주차: 정답도 20~30%대로 낮게 나올 수 있음)
- 원본 노트북의 마지막 부분처럼, 요청한 k개 중 관련 없는 이미지가 섞이면 점수로 걸러낸다

In [ ]:
import matplotlib.pyplot as plt


def search(query, k=TOP_K):
    q = embed_texts([query])
    faiss.normalize_L2(q)
    scores, ids = index.search(q, k)
    probs = 1 / (1 + np.exp(-(T * scores[0] + B)))
    return [{"id": int(i), "path": meta[i]["path"], "cos": float(s), "prob": float(p),
             "caption": (meta[i]["captions"] or [""])[0]}
            for i, s, p in zip(ids[0], scores[0], probs)]


def show(query, k=TOP_K):
    hits = search(query, k)
    print(f"🔎 {query}")
    for r, h in enumerate(hits, 1):
        mark = "  " if h["prob"] >= MIN_PROB else "✗ "  # 확률이 낮으면 관련 없음 표시
        print(f"  {mark}{r}위  id={h['id']:<5} 코사인 {h['cos']:.3f} · 짝일 확률 {h['prob']:.1%}  | {h['caption'][:60]}")
    fig, axes = plt.subplots(1, len(hits), figsize=(3 * len(hits), 3.2))
    for ax, (r, h) in zip(np.atleast_1d(axes), enumerate(hits, 1)):
        ax.imshow(Image.open(h["path"]))
        ax.set_title(f"#{r} cos {h['cos']:.2f} / p {h['prob']:.0%}", fontsize=9)  # Colab matplotlib엔 한글 폰트가 없어서 영어로
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return hits


show("a dog running on the grass");

In [ ]:
# 같은 내용을 한국어로 → 다국어 모델이면 비슷한 이미지가 나와야 한다
show("잔디밭에서 뛰는 강아지");
show("바다에서 노는 아이들");
show("빨간 옷을 입은 사람");

## 5. 인덱스 다시 불러오기
런타임이 끊긴 뒤에도 이미지를 다시 임베딩할 필요 없이, 저장해 둔 인덱스와 메타데이터만 불러오면 바로 검색할 수 있다. (모델은 질의를 임베딩하려고 여전히 필요)

In [ ]:
index = faiss.read_index(INDEX_PATH)
saved = json.load(open(META_PATH))
meta = saved["items"]
print(f"불러옴: {index.ntotal}개 · 모델 {saved['model']} · 데이터 {saved['dataset']}")
assert saved["model"] == MODEL_ID, "인덱스를 만든 모델과 지금 모델이 다르면 검색 결과가 엉망이 된다"
show("a cat sitting on a sofa", 3);

## 6. (선택) 검색 정확도: Recall@K
캡션이 있는 데이터셋이면, **캡션을 질의로 넣었을 때 그 캡션의 원래 이미지가 상위 K장 안에 드는 비율**을 잰다.
3주차 RAGAS처럼 설정(모델, 질의 형식 등)을 바꿨을 때 숫자로 비교하는 용도.

In [ ]:
def recall_at_k(ks=(1, 5, 10), prompt="{}"):
    pairs = [(i, prompt.format(m["captions"][0])) for i, m in enumerate(meta) if m["captions"]]
    if not pairs:
        print("캡션이 있는 이미지가 없어서 Recall@K를 계산할 수 없습니다.")
        return None
    q = np.vstack([embed_texts([t for _, t in pairs[j:j + BATCH]]) for j in range(0, len(pairs), BATCH)])
    faiss.normalize_L2(q)
    _, ids = index.search(q, max(ks))
    gold = np.array([i for i, _ in pairs])[:, None]
    result = {f"R@{k}": float((ids[:, :k] == gold).any(axis=1).mean()) for k in ks}
    print(f"질의 {len(pairs)}개 · 형식 '{prompt}' →", "  ".join(f"{k} {v:.1%}" for k, v in result.items()))
    return result


recall_at_k()
recall_at_k(prompt="This is a photo of {}.")  # 4주차: pipeline이 쓰는 문장 형식으로 감싸면 달라질까?

## 7. (선택) 검색 데모 화면 (Gradio)
실행하면 `https://xxxx.gradio.live` 주소가 나온다. 문장을 입력하면 확률 높은 순으로 이미지를 보여준다.

In [ ]:
!pip install -q gradio
import gradio as gr


def ui_search(query, k, min_prob):
    if not query.strip():
        return [], "찾을 내용을 문장으로 입력해 주세요."
    hits = search(query, int(k))
    shown = [h for h in hits if h["prob"] >= min_prob]
    info = f"상위 {len(hits)}장 중 짝일 확률 {min_prob:.0%} 이상 {len(shown)}장"
    return [(h["path"], f"코사인 {h['cos']:.2f} · 확률 {h['prob']:.1%}") for h in (shown or hits)], info


with gr.Blocks(title="DRAG 5주차 이미지 검색") as demo:
    gr.Markdown(f"# 🔎 텍스트 → 이미지 검색 (SigLIP + FAISS)\n`{MODEL_ID}` · `{DATASET_ID}` {index.ntotal}장")
    with gr.Row():
        q_in = gr.Textbox(label="찾을 이미지를 문장으로", placeholder="예: 잔디밭에서 뛰는 강아지", scale=6)
        k_in = gr.Slider(1, 20, value=8, step=1, label="결과 수", scale=2)
        p_in = gr.Slider(0.0, 0.5, value=MIN_PROB, step=0.01, label="최소 짝일 확률", scale=2)
    btn = gr.Button("검색", variant="primary")
    info = gr.Markdown()
    gallery = gr.Gallery(columns=4, height=520)
    gr.Examples(["a dog running on the grass", "잔디밭에서 뛰는 강아지", "바다에서 노는 아이들", "빨간 옷을 입은 사람"], inputs=q_in)
    for ev in (q_in.submit, btn.click):
        ev(ui_search, [q_in, k_in, p_in], [gallery, info])

demo.queue().launch(share=True, debug=False)